# 05 · Hands for J.A.R.V.I.S. – tools

A model can only produce text. **Tools** let an agent *do* things: run code, search files, search the web or call your own functions. The model decides **when** to call a tool; something else **executes** it.

| Tool | Who executes it | Example here |
|---|---|---|
| **Function** | your code (you run it and send the result back) | arc reactor output |
| **Code interpreter** | Foundry sandbox | math and charts |
| **File search** | Foundry (vector store) | the workshop handbook |
| **Web search** | Foundry | current facts |
| **MCP / A2A** | a remote server / agent | chapters 06 and 17 |

Deep dive: [docs/tools.md](../docs/tools.md).

In [ ]:
import sys; sys.path.insert(0, '.')
from stark_helpers import *
from stark_replies import show_reply

import json
from azure.ai.projects.models import PromptAgentDefinition, FunctionTool, CodeInterpreterTool, FileSearchTool

project_openai = openai_client()     # OpenAI client on the project endpoint (Responses, Conversations, vector stores)
created_agents = []                  # agent names, deleted in the clean-up

## 1 · Function calling – you run the code
1. Describe the function with a JSON schema.
2. The model answers with a `function_call` item (name + arguments) instead of text.
3. **You** run it and send back a `function_call_output`. The model then writes the final answer.

Every agent in this chapter is a `PromptAgentDefinition` with a `tools=[...]` list: the tools are part of the version. Its name goes into `created_agents` for the clean-up. `show_reply()` ([stark_replies.py](stark_replies.py)) shows each turn as Markdown: the prompt, the whole answer and the tool calls.

### Step 1 · The function and its description
The Python function is yours and runs only here. The model never sees its code, only the `FunctionTool`: name, description and parameter schema.

In [ ]:
def get_arc_reactor_output(suit: str) -> dict:
    data = {"mark 3": 3.0, "mark 42": 8.5, "mark 85": 42.0}
    return {"suit": suit, "output_gw": data.get(suit.lower(), None)}

REGISTRY = {"get_arc_reactor_output": get_arc_reactor_output}   # tool name -> the Python function that runs it

reactor_tool = FunctionTool(
    name="get_arc_reactor_output",
    description="Get the arc reactor power output in gigawatts for an Iron Man suit.",
    parameters={"type": "object", "properties": {"suit": {"type": "string", "description": "e.g. 'mark 42'"}},
                "required": ["suit"], "additionalProperties": False},
    strict=True,
)
reactor_agent = project().agents.create_version(
    agent_name="jarvis-nb05-fn",
    definition=PromptAgentDefinition(model=model("chat"), instructions="You are J.A.R.V.I.S. Use tools for facts. Be brief.",
                                     tools=[reactor_tool]),
)
created_agents.append(reactor_agent.name)
print(reactor_agent.name, "version", reactor_agent.version)

### Step 2 · The model asks, it does not answer
**Goal:** see that the first response holds no answer, only requests to call your function.

**Expected:** `function_call` items for `get_arc_reactor_output`, most likely one per suit (`mark 3` and `mark 42`), and no message text yet.

The cell opens a new conversation, `reactor_conversation`. Re-running it starts over: an old conversation with unanswered calls would refuse new input.

In [ ]:
reactor_question = "Compare the reactor output of the Mark 3 and Mark 42."
reactor_conversation = project_openai.conversations.create()
reactor_response = project_openai.responses.create(conversation=reactor_conversation.id, input=reactor_question,
                                                   extra_body=agent_ref(reactor_agent))
print(f"> {reactor_question}\n")
for item in reactor_response.output:
    print(item.type, getattr(item, "name", ""), getattr(item, "arguments", ""))

### Step 3 · The function-call loop
**You** run each requested call and send the results back as `function_call_output` items, in the same conversation. The model may ask again after seeing the results, so loop until no `function_call` is left, and cap the rounds.

**Goal:** turn the calls from step 2 into an answer.

**Expected:** one `tool call:` line per call from step 2 (Mark 3: `3.0`, Mark 42: `8.5` GW), then an answer that compares 3.0 GW with 8.5 GW. Run step 2 first: the loop continues `reactor_response`.

In [ ]:
for _ in range(5):          # cap the rounds: a confused model could loop forever
    calls = [i for i in reactor_response.output if i.type == "function_call"]
    if not calls:
        break
    outputs = []
    for call in calls:
        args = json.loads(call.arguments)
        result = REGISTRY[call.name](**args)
        print(f"  tool call: {call.name}({args}) -> {result}")
        outputs.append({"type": "function_call_output", "call_id": call.call_id, "output": json.dumps(result)})
    reactor_response = project_openai.responses.create(conversation=reactor_conversation.id, input=outputs,
                                                       extra_body=agent_ref(reactor_agent))

final = final_message(reactor_response)
show_reply(reactor_question, reactor_response, checks={
    "mentions 3.0 GW (Mark 3)": mark("3.0" in final or "3 GW" in final),
    "mentions 8.5 GW (Mark 42)": mark("8.5" in final),
})

> 💡 From chapter 13 on, `run_functions(...)` in [stark_helpers.py](stark_helpers.py) runs this same loop for you.

## 2 · Code interpreter – a sandbox that runs Python
No code from you: the model writes Python, Foundry runs it in a sandbox and returns the result (and files).

**Goal:** exact arithmetic, done by code instead of the model.

**Expected:** 3.7 × 365.25 × 86,400 = **116,763,120** seconds, a `code_interpreter_call` item, and the code the model wrote.

In [ ]:
calc_agent = project().agents.create_version(
    agent_name="jarvis-nb05-ci",
    definition=PromptAgentDefinition(model=model("chat"), instructions="You are J.A.R.V.I.S. For any calculation, write and run Python.",
                                     tools=[CodeInterpreterTool()]),
)
created_agents.append(calc_agent.name)

seconds_question = "How many seconds are in 3.7 years (365.25 days)? Show the exact number."
calc_reply = project_openai.responses.create(input=seconds_question, extra_body=agent_ref(calc_agent))
digits = "".join(ch for ch in final_message(calc_reply) if ch.isdigit() or ch == ".")
show_reply(seconds_question, calc_reply, checks={"answer has 116,763,120": mark("116763120" in digits)})
for item in calc_reply.output:
    if item.type == "code_interpreter_call":
        print("--- code the model wrote ---\n" + (item.code or ""))

## 3 · File search – ask your own documents
Upload a file to a **vector store**; the agent retrieves matching passages and answers from them (RAG without building a pipeline). The file is [data/workshop-handbook.md](../data/workshop-handbook.md).

Each run of the next cell creates a new vector store, `handbook_store`; the clean-up deletes the last one.

In [ ]:
handbook_store = project_openai.vector_stores.create(name="stark-handbook-nb05")
with open(ROOT / "data" / "workshop-handbook.md", "rb") as f:
    project_openai.vector_stores.files.upload_and_poll(vector_store_id=handbook_store.id, file=f)
print("vector store:", handbook_store.id)

handbook_agent = project().agents.create_version(
    agent_name="jarvis-nb05-fs",
    definition=PromptAgentDefinition(model=model("chat"), instructions="Answer only from the handbook. If it is not there, say so.",
                                     tools=[FileSearchTool(vector_store_ids=[handbook_store.id])]),
)
created_agents.append(handbook_agent.name)

### Case A · The answer is in the handbook
**Goal:** an answer taken from your document, not from the model's memory.

**Expected:** Pepper Potts, from the line "Pepper Potts approves all purchases above 5,000 USD.", and a `file_search_call` item.

In [ ]:
purchase_question = "Who approves purchases above 5,000 USD?"
purchase_reply = project_openai.responses.create(input=purchase_question, extra_body=agent_ref(handbook_agent))
show_reply(purchase_question, purchase_reply, checks={"names Pepper Potts": mark("Pepper" in final_message(purchase_reply))})

### Case B · The answer is not in the handbook
**Goal:** same agent, a question the handbook cannot answer. The instructions say to admit it.

**Expected:** the agent says the handbook does not contain the Wi-Fi password and does not make one up. The handbook has no Wi-Fi entry, so any password in the answer is invented.

In [ ]:
wifi_question = "What is the Wi-Fi password?"
wifi_reply = project_openai.responses.create(input=wifi_question, extra_body=agent_ref(handbook_agent))
show_reply(wifi_question, wifi_reply)

## 4 · Web search
> ⚠️ Web search sends the query to Bing outside the Azure compliance boundary. Read the data-handling terms before using it with customer data. May not be available in every region.

**Expected:** ❓ an answer about a recent announcement with cited sources, and a web search item in the tool calls. The answer changes from day to day. Where web search is not available, the cell prints why instead.

In [ ]:
from azure.ai.projects.models import WebSearchPreviewTool

news_question = "What is the latest Microsoft Foundry announcement?"
try:
    web_agent = project().agents.create_version(
        agent_name="jarvis-nb05-ws",
        definition=PromptAgentDefinition(model=model("chat"), instructions="Search the web for current facts and cite sources.",
                                         tools=[WebSearchPreviewTool()]),
    )
    created_agents.append(web_agent.name)
    news_reply = project_openai.responses.create(input=news_question, extra_body=agent_ref(web_agent))
    show_reply(news_question, news_reply)
except Exception as exc:
    print("web search not available here:", type(exc).__name__, str(exc)[:200])

## 👀 In the portal
**Build → Agents → jarvis-nb05-fs**: the agent with the file search tool and its vector store `stark-handbook-nb05`. Open the other `jarvis-nb05-*` agents to compare their tools. Try the **Playground**.

> 💡 All pages: [docs/portal-tour.md](../docs/portal-tour.md)

## 🧹 Clean up

In [ ]:
project_openai.conversations.delete(reactor_conversation.id)
project_openai.vector_stores.delete(handbook_store.id)
for name in created_agents:
    delete_agent(name)

## 🎯 Recap
Pick the tool by **who should run it**: your code (function), Foundry (code interpreter, file search, web search) or someone else's server (MCP, A2A).

> 💡 Next: [06 · MCP](06_mcp_tools.ipynb) – tools you don't have to write.